# 3. Perfil taxonômico a partir de dados shotgun 

Este notebook contém códigos em Python para processar as saídas geradas pelo MetaPhlAn.

O objetivo é verificar se os arquivos de abundância taxonômica foram corretamente carregados no ambiente do Galaxy Notebook e, em seguida, prepará-los para visualizações e análises posteriores. 

Execute cada célula do notebook clicando no ícone de execução (`>`) ou utilizando o atalho correspondente.

In [ ]:
# Importando as bibliotecas que serão utilizadas ao longo do notebook 
import os 
import glob

In [ ]:
''' 
Esta célula verifica se os arquivos de saída do MetaPhlAn foram corretamente reconhecidos pelo Galaxy Notebook. 
Ao executar esta célula, devem aparecer os quatro arquivos com extensão .tabular, correspondentes às amostras analisadas.
'''

# Percorre a pasta padrão onde o Galaxy disponibiliza os arquivos de entrada do notebook
for root, dirs, files in os.walk("galaxy_inputs"):
    print(root)
    # Lista os arquivos encontrados dentro de cada pasta
    for f in files:
        print("  ", f)

galaxy_inputs
   galaxy_inputs_raw.json
   galaxy_inputs.json
galaxy_inputs/roteiro7
   MetaPhlAn_HSM6XRRV:taxon_abundances.tabular
   MetaPhlAn_HSM7CYYP:taxon_abundances.tabular
   MetaPhlAn_HSM5MD6Q:taxon_abundances.tabular
   MetaPhlAn_HSMA33KQ:taxon_abundances.tabular


In [ ]:
# Localiza automaticamente todos os arquivos .tabular gerados pelo MetaPhlAn
# dentro da pasta padrão de entrada do Galaxy Notebook.
input_files = glob.glob("galaxy_inputs/*/*.tabular", recursive=True)

def convert_metaphlan_to_krona(infile, output_dir="outputs"):
    """
    Converte uma tabela de abundância taxonômica gerada pelo MetaPhlAn
    para um arquivo compatível com o Krona.

    O MetaPhlAn representa cada táxon como uma hierarquia separada por '|',
    por exemplo:
        k__Bacteria|p__Firmicutes|c__Bacilli

    O Krona espera um formato em que a primeira coluna seja a abundância
    e as colunas seguintes representem os níveis taxonômicos separados:
        abundância    Bacteria    Firmicutes    Bacilli
    """

    # Define o nome do arquivo de saída a partir do nome do arquivo de entrada.
    # Exemplo:
    # MetaPhlAn_HSM7CYYP:taxon_abundances.tabular -> MetaPhlAn_HSM7CYYP:taxon_krona.tsv
    sample_name = os.path.basename(infile).replace("_abundances.tabular", "_krona.tsv")
    outfile = os.path.join(output_dir, sample_name)

    # Lista que armazenará as linhas já convertidas para o formato do Krona.
    rows_out = []

    # Abre a tabela de saída do MetaPhlAn.
    with open(infile) as f:
        for line in f:
            line = line.strip()

            # Ignora linhas vazias e linhas de comentário/cabeçalho.
            if not line or line.startswith("#"):
                continue

            # Divide a linha em colunas.
            fields = line.split("\t")

            # Garante que a linha tenha pelo menos as colunas esperadas.
            if len(fields) < 3:
                continue

            # Na saída do MetaPhlAn, a primeira coluna contém a linhagem taxonômica
            # e a terceira coluna contém a abundância relativa.
            clade = fields[0]
            abundance = fields[2]

            # Remove a categoria UNCLASSIFIED, pois ela não representa
            # uma linhagem taxonômica hierárquica.
            if clade == "UNCLASSIFIED":
                continue

            # Separa os níveis taxonômicos e remove os prefixos
            # usados pelo MetaPhlAn, como k__, p__, c__, o__, f__, g__ e s__.
            taxa = []
            for level in clade.split("|"):
                if "__" in level:
                    taxa.append(level.split("__", 1)[1])
                else:
                    taxa.append(level)

            # Adiciona a linha no formato esperado pelo Krona:
            # abundância + níveis taxonômicos.
            rows_out.append([abundance] + taxa)

    # Escreve o arquivo final em formato TSV.
    with open(outfile, "w") as out:
        for row in rows_out:
            out.write("\t".join(row) + "\n")

    return outfile

In [ ]:
# Aplica a função a cada arquivo .tabular encontrado.
for infile in input_files:
    outfile = convert_metaphlan_to_krona(infile)
    print(f"Arquivo gerado: {outfile}")

In [ ]:
'''
ARQUIVO ANTES:
'''
!head outputs/*HSM6XRRV:taxon_abundances.tabular

In [ ]:
'''
ARQUIVO AGORA:
'''
!head outputs/*HSM6XRRV:taxon_krona.tsv

In [ ]:
# Envia os arquivos convertidos para o histórico do Galaxy. 
# Depois de executar esta célula, os arquivos *_krona.tsv deverão aparecer
# no painel History, podendo ser usados como entrada na ferramenta Krona.

for f in glob.glob("outputs/*_krona.tsv"):
    put(f)